# Bayesian Search — ResNet-101

Otimização bayesiana de hiperparâmetros usando **Optuna** para o modelo ResNet-101.

Espaço de busca:
- Dropout (0.2-0.5), Neurônios FC (256/512/1024/4096)
- Optimizer (Adam/SGD), Learning Rate (1e-5 a 1e-2)
- Batch Size (32/64/128), Activation (ReLU/LeakyReLU)

In [ ]:
import os, torch, numpy as np, torch.nn as nn, torch.optim as optim
from torchvision import datasets, models, transforms
from ignite.engine import Engine, Events
from ignite.handlers import EarlyStopping
from ignite.metrics import Accuracy, Loss
import optuna
from dotenv import load_dotenv
load_dotenv()

DATASET_PATH = os.getenv("DATASET_PATH", "/caminho/para/DADOS-DIVIDIDOS")
PRETRAINED_WEIGHTS = os.getenv("RESNET101_PRETRAINED", "/caminho/para/models/ResNet_101_ImageNet_plant-model-84.pth")
FEATURE_EXTRACT = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.RandomVerticalFlip(),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224), transforms.CenterCrop(224),
        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(224), transforms.CenterCrop(224),
        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

image_datasets = {x: datasets.ImageFolder(os.path.join(DATASET_PATH, x), data_transforms[x]) for x in ['train', 'val', 'test']}

In [ ]:
def set_parameter_requires_grad(model, fe):
    if fe:
        for p in model.parameters(): p.requires_grad = False

def objective(trial):
    # --- Espaço de busca ---
    dropout1 = trial.suggest_float("dropout1", 0.2, 0.5)
    dropout2 = trial.suggest_float("dropout2", 0.2, 0.5)
    activation_name = trial.suggest_categorical("activation", ["ReLU", "LeakyReLU"])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    fc1 = trial.suggest_categorical("num_neurons_fc1", [256, 512, 1024, 4096])
    fc2 = trial.suggest_categorical("num_neurons_fc2", [256, 512, 1024, 4096])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    momentum = trial.suggest_float("momentum", 0.7, 0.99) if optimizer_name == "SGD" else None
    
    activation = nn.ReLU() if activation_name == "ReLU" else nn.LeakyReLU()
    
    # --- Modelo ---
    model = models.resnet101(pretrained=False)
    set_parameter_requires_grad(model, FEATURE_EXTRACT)
    state_dict = torch.load(PRETRAINED_WEIGHTS, map_location=device)
    del state_dict['fc.weight']; del state_dict['fc.bias']
    model.load_state_dict(state_dict, strict=False)
    
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout1), nn.Linear(num_features, fc1), activation,
        nn.Dropout(p=dropout2), nn.Linear(fc1, fc2), activation,
        nn.Linear(fc2, 2),
    )
    model = model.to(device)
    
    # --- DataLoaders ---
    dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=batch_size, shuffle=(x=='train'), num_workers=4) for x in ['train', 'val']}
    
    # --- Optimizer ---
    criterion = nn.CrossEntropyLoss()
    if optimizer_name == "Adam":
        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    else:
        optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, momentum=momentum)
    
    # --- Ignite ---
    def train_step(engine, batch):
        model.train(); x, y = batch[0].to(device), batch[1].to(device)
        optimizer.zero_grad(); out = model(x); loss = criterion(out, y)
        loss.backward(); optimizer.step(); return loss.item(), out, y
    
    def eval_step(engine, batch):
        model.eval()
        with torch.no_grad(): x, y = batch[0].to(device), batch[1].to(device); return model(x), y
    
    trainer = Engine(train_step); evaluator = Engine(eval_step)
    Accuracy().attach(evaluator, 'accuracy')
    
    def score_fn(e): return e.state.metrics['accuracy']
    evaluator.add_event_handler(Events.COMPLETED, EarlyStopping(patience=10, score_function=score_fn, trainer=trainer))
    
    best_acc = [0.0]
    @trainer.on(Events.EPOCH_COMPLETED)
    def log(engine):
        evaluator.run(dataloaders['val'])
        acc = evaluator.state.metrics['accuracy']
        best_acc[0] = max(best_acc[0], acc)
        print(f"Val Accuracy: {acc:.4f}")
    
    trainer.run(dataloaders['train'], max_epochs=100)
    return best_acc[0]

print("Função objetivo definida.")

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print(f"\n{'='*50}")
print(f"Melhor trial: {study.best_trial.number}")
print(f"Melhor accuracy: {study.best_trial.value:.4f}")
print(f"Melhores hiperparâmetros:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")